# Saving and Reapplying Clustering Results

This notebook demonstrates the **cluster once, apply many times** workflow using `ClusteringResult`.

**Common use cases:**
1. **Cluster on a subset of variables** (e.g., wind only), then apply that clustering to all variables
2. **Save clustering to file**, reload later, and apply to updated or different data
3. **Share clustering** across different datasets or team members

Author: Maximilian Hoffmann

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd

# Configure Plotly for sphinx/nbsphinx output
import plotly.io as pio

import tsam
from tsam import ClusterConfig

pio.renderers.default = "notebook"

# Ensure results directory exists
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

### Input data 

Read in time series from testdata.csv with pandas

In [ ]:
raw = pd.read_csv("testdata.csv", index_col=0)

Show a slice of the dataset

In [ ]:
raw.head()

Show the shape of the raw input data: 4 types of timeseries (GHI, Temperature, Wind and Load) for every hour in a year

In [ ]:
raw.shape

Plot an example series - in this case the wind speed

In [ ]:
# Original wind heatmap
tsam.plot.heatmap(raw, column="Wind", period_hours=24, title="Original Wind")

## Step 1: Initial Clustering

First, let's perform a standard aggregation on all variables to establish a baseline.

Initialize an aggregation class object with hierarchical clustering as method for eight typical days, without any integration of extreme periods. Alternative clusterMethod's are 'averaging','hierarchical' and 'k_medoids'.

In [ ]:
result = tsam.aggregate(
    raw,
    n_periods=8,
    period_hours=24,
    cluster=ClusterConfig(method="hierarchical", representation="medoid"),
)

Create the typical periods

In [ ]:
typical_periods = result.typical_periods

In [ ]:
typical_periods

Show shape of typical periods: 4 types of timeseries for 8*24 hours

In [ ]:
typical_periods.shape

Repredict the original time series based on the typical periods

In [ ]:
reconstructed = result.reconstruct()

Plot the repredicted data

In [ ]:
# Predicted wind heatmap (all attributes)
tsam.plot.heatmap(
    reconstructed,
    column="Wind",
    period_hours=24,
    title="Predicted Wind (All Attributes)",
)

## Use Case 1: Cluster on Subset, Apply to All

A common workflow is to cluster on a subset of variables (e.g., wind only) and then apply that clustering to all variables. This ensures the clustering is optimized for the most important variable while still getting aggregated values for everything.

Clustering the solar time series only with 8 typical days and hierarchical clustering leads to different typical days in another sequence.

Isolate wind time series and show first lines of data

In [ ]:
raw_wind = raw.loc[:, "Wind"].to_frame()
raw_wind.head()

Now same clustering procedure as above for the isolated wind time series

In [ ]:
result_wind = tsam.aggregate(
    raw_wind,
    n_periods=8,
    period_hours=24,
    cluster=ClusterConfig(method="hierarchical", representation="medoid"),
)

In [ ]:
typical_periods_wind = result_wind.typical_periods

Export for preprocess time series for testing

In [ ]:
# Export preprocessed time series for testing/debugging.
# Note: The _aggregation attribute is internal and may change in future versions.
# This access pattern is appropriate for testing but should not be relied upon in production code.
result_wind._aggregation.normalizedPeriodlyProfiles.to_csv(
    RESULTS_DIR / "preprocessed_wind.csv"
)

In [ ]:
typical_periods_wind.shape

In [ ]:
reconstructed_wind = result_wind.reconstruct()

In [ ]:
# Predicted wind heatmap (wind only)
tsam.plot.heatmap(
    reconstructed_wind,
    column="Wind",
    period_hours=24,
    title="Predicted Wind (Wind Only)",
)

When we compare both plots, we see that 8 typical periods for wind only can better account extreme periods, but the cluster order in general changes

In [ ]:
result.cluster_assignments

In [ ]:
result_wind.cluster_assignments

### Applying Wind Clustering to All Variables

Now we use `ClusteringResult.apply()` to transfer the wind-only clustering to all attributes. This is the key feature: **cluster once, apply many times**.

In [ ]:
# Apply wind-only clustering to all attributes
result_predef = result_wind.clustering.apply(raw)

In [ ]:
typical_periods_predef = result_predef.typical_periods

In [ ]:
typical_periods_predef.shape

Save typical periods to .csv file

In [ ]:
typical_periods_predef.to_csv(RESULTS_DIR / "testperiods_predef_cluster_order.csv")

In [ ]:
reconstructed_predef = result_predef.reconstruct()

Now we compare the cluster orders

In [ ]:
result_wind.cluster_assignments

In [ ]:
result_predef.cluster_assignments

As it can be seen, the cluster order for the four attributes is now identical to the cluster order of the wind time series clustering. Using `ClusteringResult.apply()` transfers both the cluster assignments AND the cluster centers, ensuring the exact same representative periods are used.

In [ ]:
# Predicted wind heatmap (predefined cluster order)
tsam.plot.heatmap(
    reconstructed_predef,
    column="Load",
    period_hours=24,
    title="Predicted Wind (Predefined Cluster Order)",
)

The wind values in the transferred result match exactly because `ClusteringResult.apply()` transfers:
- **Cluster assignments**: Which days belong to which cluster
- **Cluster centers**: Which original day represents each cluster
- **Representation method**: How typical periods are computed (mean, medoid, etc.)
- **Rescale setting**: Whether to rescale to preserve original means

### Transferring with Segmentation

The transfer also works with segmentation. All segment information is preserved:

In [ ]:
from tsam import ClusteringResult, SegmentConfig

# Aggregate wind with segmentation
result_wind_seg = tsam.aggregate(
    raw_wind,
    n_periods=8,
    period_hours=24,
    cluster=ClusterConfig(method="hierarchical", representation="medoid"),
    segments=SegmentConfig(n_segments=6),
)

# The clustering property contains all transfer state
result_wind_seg.clustering

In [ ]:
# Transfer everything with the apply() method
result_full_transfer = result_wind_seg.clustering.apply(raw)

# The result has the same segmented structure
result_full_transfer.typical_periods

## Use Case 2: Save and Load Clustering

The `ClusteringResult` can be saved to JSON and loaded later. This enables:
- **Reproducibility**: Store clustering configuration for later use
- **Sharing**: Send clustering results to colleagues
- **Batch processing**: Apply the same clustering to multiple datasets

In [ ]:
# Save to JSON (simple method)
result_wind_seg.clustering.to_json(str(RESULTS_DIR / "clustering.json"))

# Load from JSON
clustering_loaded = ClusteringResult.from_json(str(RESULTS_DIR / "clustering.json"))

# Use loaded clustering
result_from_file = clustering_loaded.apply(raw)
result_from_file.typical_periods.head()

### Unified Assignments DataFrame

The `assignments` property provides a convenient DataFrame with all assignment information for each original timestep:

In [ ]:
# Unified assignments DataFrame
result_full_transfer.assignments.head(30)

## Use Case 3: Apply to Different Data

A key feature is applying saved clustering to completely different data - for example, a different year or updated forecasts. The clustering structure (which days belong to which cluster) remains the same, but the values come from the new data.

In [ ]:
# Simulate a "different year" by scaling the data
# In practice, this would be actual data from another year
raw_year2 = raw.copy()
raw_year2["Wind"] = raw_year2["Wind"] * 1.1  # 10% more wind
raw_year2["Load"] = raw_year2["Load"] * 0.95  # 5% less load

# Apply the SAME clustering to the new data
result_year2 = clustering_loaded.apply(raw_year2)

print("Original year wind mean:", raw["Wind"].mean())
print("Year 2 wind mean:", raw_year2["Wind"].mean())
print()
print("Typical periods from original:", result_from_file.typical_periods["Wind"].mean())
print("Typical periods from year 2:", result_year2.typical_periods["Wind"].mean())

The same 8 cluster assignments are used, but the typical period values reflect the new data. This is useful for:
- Comparing different scenarios with consistent time structure
- Energy system optimization where the time structure must match across scenarios
- Sensitivity analysis with modified input data

## Summary

This notebook demonstrated the **cluster once, apply many times** workflow:

1. **`result.clustering`** - Access the `ClusteringResult` from any aggregation result
2. **`clustering.apply(data)`** - Apply clustering to any compatible data
3. **`clustering.to_json(path)`** - Save clustering to file
4. **`ClusteringResult.from_json(path)`** - Load clustering from file

The `ClusteringResult` preserves all state needed for deterministic transfer:
- Cluster assignments and centers
- Segmentation structure (if used)
- Representation method (mean, medoid, etc.)
- Rescale settings